# A CNN from scratch, end to end

Booster LEAP, Computer Vision, topic 2. How does a convolution layer learn, and how do we build and check a small CNN?

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). It needs no GPU and no downloads: the data is made in the code or ships with scikit-learn. Change one thing at a time and run again; the exercises at the end tell you what to try.

## Stage 1: learn a kernel by gradient descent

Eight random 6x6 images are convolved with the vertical-edge kernel to make target feature maps. A kernel that starts at all zeros must find those nine numbers by itself. The loss is the mean squared difference between the feature map the kernel makes and the target. The gradient for one kernel weight is the sum, over all output positions, of 2 x error x the pixel that this weight touched.

1. The first printed line checks the hand-written gradient against a finite difference: the two numbers should agree.
2. Then 200 gradient-descent steps run and the loss is printed as it falls.
3. At the end the learned kernel is printed: it should look like the target, negative on the left, near 0 in the middle, positive on the right.

In [ ]:
import random

rng = random.Random(0)
imgs = [[[round(rng.random(), 2) for _ in range(6)] for _ in range(6)] for _ in range(8)]   # 8 random 6x6 images
KV = [[-1, 0, 1]] * 3                                                                     # the kernel we want the network to find

def conv(img, k):
    return [[sum(img[i + a][j + b] * k[a][b] for a in range(3) for b in range(3)) for j in range(4)] for i in range(4)]

targets = [conv(im, KV) for im in imgs]          # what a perfect kernel would output

def loss_and_grad(k):
    g = [[0.0] * 3 for _ in range(3)]
    total = 0.0
    n = len(imgs) * 16
    for im, t in zip(imgs, targets):
        out = conv(im, k)
        for i in range(4):
            for j in range(4):
                e = out[i][j] - t[i][j]
                total += e * e
                for a in range(3):
                    for b in range(3):
                        g[a][b] += 2 * e * im[i + a][j + b] / n      # d loss / d kernel[a][b]
    return total / n, g

# 1. gradient check: compare one analytic gradient with a finite difference
k = [[0.1 * (a + b) for b in range(3)] for a in range(3)]
_, g = loss_and_grad(k)
eps = 1e-6
kp = [row[:] for row in k]; kp[1][2] += eps
km = [row[:] for row in k]; km[1][2] -= eps
numeric = (loss_and_grad(kp)[0] - loss_and_grad(km)[0]) / (2 * eps)
print("gradient check:", round(g[1][2], 5), round(numeric, 5))

# 2. gradient descent from a zero kernel
k = [[0.0] * 3 for _ in range(3)]
for step in range(201):
    loss, g = loss_and_grad(k)
    if step % 50 == 0:
        print("step", step, "loss", round(loss, 5))
    k = [[k[a][b] - 0.1 * g[a][b] for b in range(3)] for a in range(3)]
print("learned kernel:")
for row in k:
    print([round(v, 2) for v in row])

gradient check: 0.82584 0.82584
step 0 loss 0.50216
step 50 loss 0.09196
step 100 loss 0.01912
step 150 loss 0.00437
step 200 loss 0.00108
learned kernel:
[-0.95, 0.01, 0.97]
[-0.95, -0.02, 0.91]
[-0.96, -0.0, 1.0]


This is what "training a CNN" means: the kernel numbers are parameters, the feature maps are computed from them, and gradient descent moves them so that the loss falls. In a real network the target is a class label and the gradient flows back through many layers, but the update of each kernel weight is the same sum of errors times the pixels it saw.

## Stage 2: a small CNN on 8x8 digits

scikit-learn contains 1,797 digit images of 8x8 pixels. The network has two blocks of convolution, batch normalization, ReLU and max pooling, then one linear layer. Read the comments for the shape after every block, and compare the printed parameter count with the count you can do by hand: (1x9+1)x8 + 16 + (8x9+1)x16 + 32 + 64x10 + 10.

In [ ]:
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
X, y = load_digits(return_X_y=True)                                   # 1,797 images of 8x8 pixels, values 0 to 16 (ships with scikit-learn)
X = torch.tensor(X / 16.0, dtype=torch.float32).reshape(-1, 1, 8, 8)    # (batch, channels, height, width)
y = torch.tensor(y)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

net = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(), nn.MaxPool2d(2),     # 8 x 4 x 4
    nn.Conv2d(8, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),   # 16 x 2 x 2
    nn.Flatten(), nn.Linear(64, 10))
print("parameters:", sum(p.numel() for p in net.parameters()))
opt = torch.optim.Adam(net.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1, 21):
    net.train()
    order = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        idx = order[i:i + 64]
        opt.zero_grad()
        loss = loss_fn(net(Xtr[idx]), ytr[idx])
        loss.backward()
        opt.step()
    if epoch in (1, 5, 10, 20):
        net.eval()
        with torch.no_grad():
            acc = (net(Xte).argmax(1) == yte).float().mean().item()
        print(f"epoch {epoch:2d}  last batch loss {loss.item():.3f}  test accuracy {acc:.3f}")

net.eval()
with torch.no_grad():
    pred = net(Xte).argmax(1)
print("wrong test images:", int((pred != yte).sum()), "of", len(yte))
print("first-layer kernel 0:", [[round(v, 2) for v in r] for r in net[0].weight[0, 0].tolist()])

train (1437, 1, 8, 8) test (360, 1, 8, 8)
parameters: 1946
epoch  1  last batch loss 0.433  test accuracy 0.708
epoch  5  last batch loss 0.030  test accuracy 0.983
epoch 10  last batch loss 0.025  test accuracy 0.978
epoch 20  last batch loss 0.001  test accuracy 0.986
wrong test images: 5 of 360
first-layer kernel 0: [[0.05, 0.24, -0.39], [-0.26, 0.0, 0.16], [-0.06, 0.14, -0.04]]


The parameter count matches the hand calculation (1946), and the test accuracy reaches 0.986. The numbers can differ slightly on another machine.

## A shape habit that saves time

The size that goes into the linear layer is easy to get wrong by hand. Push one dummy image through the feature part and count the numbers.

In [ ]:
import torch
from torch import nn

features = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
n_flat = features(torch.zeros(1, 1, 8, 8)).numel()      # push one dummy image through and count
print("flatten size:", n_flat)
head = nn.Linear(n_flat, 10)

flatten size: 64


## Try it yourself

1. In stage 1, change the learning rate from 0.1 to 0.5, then to 1.0. What happens to the loss, and why?
2. In stage 1, start the kernel from different small numbers instead of zeros. Does it still find the target?
3. In stage 2, remove the two `nn.BatchNorm2d` layers. Does the accuracy after 5 epochs change?
4. In stage 2, change the number of filters from 8 and 16 to 4 and 8. How do the parameter count and the accuracy change?
5. If your own CNN does not learn: check the shapes with a dummy image, check that labels are integers, overfit 10 images first, and lower the learning rate.